# Day 4: Chunking, Walked Through

This notebook walks through why chunking matters, different chunking strategies,
and how to measure chunk quality. Run each cell in order. Plain Python throughout.

## Cell 1: Why chunking matters

You can't embed a 100-page document as a single vector and expect it to be useful --
it would average together dozens of unrelated topics into one blurry number.

Chunking splits a document into smaller pieces first, so each piece can be embedded
and retrieved on its own. Bad chunking (cutting mid-sentence, losing context) makes
everything downstream worse. Good chunking (complete ideas, clean boundaries) makes
retrieval and generation much better. Think of it like cutting a pizza -- a clean
slice is useful on its own; a random tear is not.

## Cell 2: Different chunking strategies

Let's implement a few approaches and see how they differ.

In [ ]:
import re

SAMPLE_DOCUMENT = """
Retrieval-Augmented Generation, or RAG, is a technique that combines search with
text generation. Instead of relying only on what a language model memorized during
training, a RAG system looks up relevant information first, then uses that
information to generate an answer.

A typical RAG pipeline has three steps. First, documents are split into chunks and
turned into embeddings. Second, the system searches for the most relevant chunks.
Third, the relevant chunks are handed to a language model to generate a final answer.
""".strip()

def chunk_by_fixed_size(text, chunk_size=150):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

def chunk_by_sentences(text, sentences_per_chunk=2):
    sentences = re.split(r'(?<=[.!?])\s+', text.replace("\n", " ").strip())
    sentences = [s for s in sentences if s]
    return [" ".join(sentences[i:i + sentences_per_chunk]) for i in range(0, len(sentences), sentences_per_chunk)]

def chunk_by_paragraphs(text):
    return [p.strip() for p in text.split("\n\n") if p.strip()]

print("Fixed-size chunks:")
for c in chunk_by_fixed_size(SAMPLE_DOCUMENT):
    print(" ", repr(c[:60]))

print("\nSentence-based chunks:")
for c in chunk_by_sentences(SAMPLE_DOCUMENT):
    print(" ", repr(c[:60]))

Notice the fixed-size chunks can cut off mid-word, while sentence-based chunks
always end cleanly.

## Cell 3: Compare on a real document

Let's measure this instead of just eyeballing it: how many chunks end mid-sentence?

In [ ]:
def ends_mid_sentence(chunk):
    stripped = chunk.strip()
    return bool(stripped) and stripped[-1] not in ".!?"

fixed_chunks = chunk_by_fixed_size(SAMPLE_DOCUMENT, chunk_size=150)
sentence_chunks = chunk_by_sentences(SAMPLE_DOCUMENT, sentences_per_chunk=2)
paragraph_chunks = chunk_by_paragraphs(SAMPLE_DOCUMENT)

for name, chunks in [("Fixed-size", fixed_chunks), ("Sentence-based", sentence_chunks), ("Paragraph-based", paragraph_chunks)]:
    broken = sum(1 for c in chunks if ends_mid_sentence(c))
    avg_size = sum(len(c) for c in chunks) / len(chunks)
    print(f"{name:16s}: {len(chunks)} chunks, avg size {avg_size:.0f} chars, {broken} end mid-sentence")

## Cell 4: Semantic chunking

Instead of mechanical rules, we can group sentences by whether they're actually
about the same topic, using embeddings (like Day 2).

In [ ]:
import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x * x for x in a))
    mag_b = math.sqrt(sum(y * y for y in b))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

# Four concepts, matched to the two paragraphs in SAMPLE_DOCUMENT (two
# concepts per paragraph). More concept dimensions than a 2-concept setup
# give the topic shift between paragraphs room to actually show up as a
# drop in similarity, instead of everything blurring into one big chunk.
CONCEPTS = {
    "rag_intro":   ["rag", "retrieval-augmented", "technique", "combines", "search", "generation"],
    "reliance":    ["relying", "memorized", "training", "looks", "approach", "generate", "answer"],
    "pipeline_steps":  ["pipeline", "steps", "documents", "split", "turned", "embeddings"],
    "pipeline_search": ["searches", "relevant", "chunks", "handed", "final"],
}

def sentence_to_embedding(sentence):
    words = set(w.strip("?.,!").lower() for w in sentence.split())
    return [float(sum(1 for w in words if w in cw)) for cw in CONCEPTS.values()]

sentences = re.split(r'(?<=[.!?])\s+', SAMPLE_DOCUMENT.replace("\n", " ").strip())
sentences = [s for s in sentences if s]

chunks = []
current = [sentences[0]]
current_embedding = sentence_to_embedding(sentences[0])

for sentence in sentences[1:]:
    emb = sentence_to_embedding(sentence)
    sim = cosine_similarity(current_embedding, emb)
    if sim >= 0.3:
        current.append(sentence)
        current_embedding = [(a + b) / 2 for a, b in zip(current_embedding, emb)]
    else:
        chunks.append(" ".join(current))
        current = [sentence]
        current_embedding = emb
chunks.append(" ".join(current))

print(f"Semantic chunking produced {len(chunks)} chunks:\n")
for i, c in enumerate(chunks, 1):
    print(f"Chunk {i}: {c}\n")

## Cell 5: Real world examples

Different document types call for different strategies:

| Document type | Best strategy | Why |
|---|---|---|
| Technical docs | By function/section | Keeps code with its explanation |
| News articles | By paragraph | Each paragraph is usually one fact |
| Legal documents | By clause/section | Splitting a clause can change its meaning |
| Research papers | By named section | Keeps Methods separate from Results |
| Support docs (FAQ) | By Q&A pair | A question needs its answer attached |

See `real_world_examples.py` in this folder for runnable code for each of these.

## Cell 6: Measure quality

Let's put numbers on "good" vs "bad" chunking, instead of just eyeballing it.

In [ ]:
def starts_mid_sentence(chunk):
    stripped = chunk.strip()
    first_letter = next((c for c in stripped if c.isalpha()), None)
    return first_letter is not None and first_letter.islower()

def complete_sentence_ratio(chunks):
    if not chunks:
        return 1.0
    complete = sum(1 for c in chunks if not starts_mid_sentence(c) and not ends_mid_sentence(c))
    return complete / len(chunks)

for name, chunks in [("Fixed-size", fixed_chunks), ("Sentence-based", sentence_chunks), ("Paragraph-based", paragraph_chunks)]:
    ratio = complete_sentence_ratio(chunks)
    print(f"{name:16s}: {ratio:.0%} of chunks are complete sentences")

## Cell 7: Key takeaways

- Chunking splits big documents into pieces small enough to embed well and specific
  enough to retrieve usefully.
- Fixed-size chunking is simple but breaks sentences and loses context.
- Sentence- and paragraph-based chunking respect natural boundaries and measurably
  produce more complete, useful chunks.
- Semantic chunking goes further, grouping sentences by actual topic similarity
  rather than mechanical rules.
- The best chunking strategy depends on the document type -- structured documents
  (code, legal text, FAQs) chunk best along their own natural boundaries.
- Chunk quality is measurable (completeness ratio, context loss score), not just a
  matter of opinion -- and it has a real, outsized impact on RAG quality overall.